# Fine-tune SmolVLA-450M on `aiengineer56/piper_dataset_1100` (Colab)

Trains the PiPER pick-into-box policy on a free Colab GPU (T4 16 GB), then
pushes the checkpoint to your Hugging Face Hub. Download it locally afterward
and run inference on your RTX 4050 with `scripts/run_piper_vla.py`.

**Before running:** `Runtime -> Change runtime type -> GPU` (T4 is fine).

Dataset is public (no auth needed to read). You only need a Hugging Face token
with **write** access to push the trained checkpoint.

In [ ]:
# 1) Confirm a GPU is attached
!nvidia-smi

In [ ]:
# 2) Install LeRobot with the SmolVLA extra (few minutes)
!pip install -q "lerobot[smolvla]"
import lerobot, torch
print("lerobot", lerobot.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# 3) Log in to Hugging Face (needs a WRITE token: hf.co/settings/tokens)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 4) EDIT THIS: your Hugging Face username (where the checkpoint will be pushed)
HF_USER = "your-username"  # <-- change me
assert HF_USER != "your-username", "Set HF_USER to your Hugging Face username first."

## 5) Train

SmolVLA trains only the action expert (vision encoder frozen by default), so a
T4 handles a small batch. Settings:
- `--batch_size=4` — raise to 8 if VRAM allows, drop to 2 if you hit OOM.
- `--steps=20000` — a usable policy appears well before this; checkpoints every
  2000 steps are pushed to the Hub, so you can stop early / resume.
- Free Colab sessions can time out (~12 h, idle limits). `push_to_hub=true`
  means progress is saved to `HF_USER/piper_smolvla` as it goes.

In [ ]:
# First run downloads the ~13 GB dataset into the Colab cache, then trains.
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=aiengineer56/piper_dataset_1100 \
  --batch_size=4 \
  --steps=20000 \
  --save_freq=2000 \
  --num_workers=2 \
  --policy.device=cuda \
  --output_dir=outputs/train/piper_smolvla \
  --job_name=piper_smolvla \
  --policy.push_to_hub=true \
  --policy.repo_id={HF_USER}/piper_smolvla \
  --wandb.enable=false

## 6) Use the checkpoint locally (on your RTX 4050)

In your WSL repo, download the trained checkpoint and run the hybrid rollout:

```bash
hf download <HF_USER>/piper_smolvla \
    --local-dir outputs/train/piper_smolvla/checkpoints/last/pretrained_model

python scripts/run_piper_vla.py \
    --ckpt outputs/train/piper_smolvla/checkpoints/last/pretrained_model \
    --steps 300 --out out/piper_vla.mp4
```

SmolVLA-450M **inference** fits comfortably in 6 GB; only training needed the
bigger GPU.